Daniel Tejada Hernandez

#Optimization. Practical Tasks

Please, execute two rows of code below at first.

In [1]:
!pip install git+https://github.com/mehalyna/cooltest.git

  Cloning https://github.com/mehalyna/cooltest.git to /tmp/pip-req-build-qhp27lr0
  Running command git clone --filter=blob:none --quiet https://github.com/mehalyna/cooltest.git /tmp/pip-req-build-qhp27lr0
  Resolved https://github.com/mehalyna/cooltest.git to commit 630c96f2d3300782279879d5d13e6c1aaabf3c75
  Preparing metadata (setup.py) ... done
  Created wheel for cooltest: filename=cooltest-26.22-py3-none-any.whl size=5447 sha256=4cfef392cdb4a53b1d7830540cff5448784f2160c90580ad3221e93a0f1756c2
  Stored in directory: /tmp/pip-ephem-wheel-cache-mx22xam8/wheels/74/9f/b6/427a02d44dc13bccb5245586783e7de2f1efb9556847d744da
Successfully built cooltest


In [2]:
from cooltest.test_cool_3 import *

Pass


# Task 1. Resource Scheduling

In the resource scheduling task, we have a set of tasks to be performed, each with its own duration and resource requirements. Additionally, we have a set of available resources with limited capacity. The goal is to assign tasks to resources in such a way that all tasks are completed within their deadlines, and the resources are utilized efficiently without exceeding their capacities.

Your  task is to define `schedule_tasks()` function that takes the following inputs:

- `tasks`: A list of tuples representing tasks, where each tuple contains (`task_name`, `duration`, `resource_requirements`).
- `resources`: A dictionary representing available resources and their capacities, where the keys are resource names, and the values are their capacities.
- `deadline`: The maximum time (`deadline`) within which all tasks must be completed.

The function `schedule_tasks` returns a dictionary representing the optimal assignment of tasks to resources along with the completion time for each task.

>**Note**: implement a simple _Greedy Scheduling algorithm_ to optimize the resource scheduling task. In this algorithm, tasks are assigned to resources in a greedy manner based on their duration and resource requirements.

In [3]:
@test_schedule_task
def schedule_tasks(tasks, resources, deadline):
    """
    Optimize resource scheduling to complete tasks within the given deadline using Greedy Scheduling.
    """
    # Inicialización
    assigned_tasks = {res: [] for res in resources}
    task_times = {}
    current_capacities = resources.copy()

    # 1. Ordenar tareas en orden ascendente según su duración
    sorted_tasks = sorted(tasks, key=lambda x: x[1])

    current_time = 0.0

    for task_name, duration, requirements in sorted_tasks:
        assigned = False

        # 2. Ordenar recursos disponibles en orden ascendente de sus capacidades restantes
        # Filtrar solo los recursos que cumplen con el requisito de la tarea actual
        available_resources = [res for res in current_capacities if res in requirements]
        sorted_resources = sorted(available_resources, key=lambda r: current_capacities[r])

        for res in sorted_resources:
            # Verificar si el recurso tiene suficiente capacidad para el requisito de la tarea
            if current_capacities[res] >= requirements[res]:
                # Asignar tarea al recurso
                assigned_tasks[res].append(task_name)
                # Actualizar la capacidad del recurso
                current_capacities[res] -= requirements[res]

                # Actualizar el tiempo de finalización de la tarea
                current_time += duration
                task_times[task_name] = current_time
                assigned = True
                break

        # Si la tarea no se pudo asignar a ningún recurso, se extiende el tiempo/deadline
        if not assigned:
            # En un enfoque greedy simple, acumulamos el tiempo igualmente
            current_time += duration
            task_times[task_name] = current_time

    # Combinar los diccionarios para el formato de retorno esperado
    result = {**assigned_tasks, **task_times}
    return result

Schedule Task  Failed



# Task 2. Vehicle Routing Problem (VRP)

The **Vehicle Routing Problem (VRP)** is a classic optimization problem that involves a fleet of vehicles tasked with delivering goods or services to a set of customers from a central depot. Each customer has a demand for a certain quantity of goods, and the vehicles have limited capacities to carry these goods. The goal is to find the optimal set of routes for the vehicles such that all customers are visited exactly once, the total demand of each route does not exceed the vehicle capacity, and the overall travel time or distance is minimized.

Your next task is to define function `optimize_vrp()` that takes the following inputs:

- `depot`: The coordinates (x, y) of the depot where all vehicles start and end their routes.
- `customers`: A list of tuples representing customer locations and their demands, where each tuple contains (x, y, demand).
- `vehicle_capacity`: The maximum capacity of each vehicle.
- `num_vehicles`: The number of vehicles available in the fleet.

The function `optimize_vrp()` returns the optimized routes for the vehicles, along with the total travel distance.

Additionally you may define the function `calculate_distance()` and use it to calculate the distance between two locations.


> **Note:** The function will `optimize_vrp()` implement a brute-force approach to solve the Vehicle Routing Problem (VRP) and find the optimized routes for a fleet of vehicles to minimize travel distance. The function takes the depot location, customer locations and demands, vehicle capacity limit, and the number of available vehicles as input and returns the optimized routes for the vehicles along with the total travel distance. It uses brute force to generate all possible permutations of customer indices and evaluates the total travel distance for each permutation to find the best solution.

In [4]:
import itertools
import math

def calculate_distance(coord1, coord2):
    """
    Calculate the Euclidean distance between two points in 2D space.
    """
    return math.sqrt((coord1[0] - coord2[0])**2 + (coord1[1] - coord2[1])**2)

@test_optimize_vrp
def optimize_vrp(depot, customers, vehicle_capacity, num_vehicles):
    """
    Optimize the Vehicle Routing Problem to minimize total travel distance using Brute Force.
    """
    best_distance = float('inf')
    best_routes = []

    # Generar todas las permutaciones posibles de los índices de los clientes
    customer_indices = list(range(len(customers)))

    for perm in itertools.permutations(customer_indices):
        routes = []
        current_route = []
        current_capacity = 0
        total_distance = 0
        valid_solution = True

        for idx in perm:
            # En el ejemplo la lista 'customers' puede ser de coordenadas directas o incluir demandas.
            # Según el docstring y el ejemplo: customers = [(1, 3), (3, 5), ...]
            # Nota: Si el ejemplo de uso no define las demandas por separado, asumimos demanda fija de 1 por cliente.
            customer_loc = customers[idx]
            demand = 1 # Valor por defecto si solo se pasan coordenadas en el ejemplo de uso

            if current_capacity + demand > vehicle_capacity or len(routes) == num_vehicles:
                if current_route:
                    routes.append(current_route)
                    current_route = []
                    current_capacity = 0
                else:
                    valid_solution = False
                    break

            current_route.append(customer_loc)
            current_capacity += demand

        if current_route:
            routes.append(current_route)

        # Si excede el número de vehículos permitidos, no es válida
        if len(routes) > num_vehicles or not valid_solution:
            continue

        # Calcular la distancia total de este conjunto de rutas
        for route in routes:
            if not route:
                continue
            # Desde el depósito al primer cliente
            total_distance += calculate_distance(depot, route[0])
            # Entre clientes de la ruta
            for i in range(len(route) - 1):
                total_distance += calculate_distance(route[i], route[i+1])
            # Regreso al depósito
            total_distance += calculate_distance(route[-1], depot)

        # Encontrar la ruta con la distancia total mínima
        if total_distance < best_distance:
            best_distance = total_distance
            best_routes = routes

    return best_routes

VRP Task  Failed



# Task 3. Inventory Management

**Inventory management** is the process of efficiently tracking and controlling the flow of goods or products in a business. The goal is to strike a balance between minimizing inventory costs and ensuring sufficient stock levels to meet customer demand. The inventory management problem involves determining the optimal inventory levels to minimize holding costs (costs associated with carrying inventory) while avoiding stockouts (running out of stock) and backorders (unfilled customer orders).

Your task is to define `optimize_inventory_management()` function that takes the following inputs:

- `demand`: A list representing the demand for each period (e.g., month, week) in the planning horizon.
- `holding_cost`: The cost of holding one unit of inventory for one period (e.g., month, week).
- `ordering_cost`: The cost of placing an order for a fixed quantity of inventory.
- `initial_inventory`: The initial inventory level at the beginning of the planning horizon.
- `reorder_point`: The inventory level at which a new order should be placed to avoid stockouts.

The function `optimize_inventory_management` should return a list representing the optimal inventory levels for each period in the planning horizon.

You have to use Linear Programming to find the optimal inventory levels for each period. The decision variables are the inventory levels and the order quantity for each period. The objective function aims to minimize the total cost, which includes both holding costs and ordering costs.

Constraints ensure that the inventory at the beginning of each period is sufficient to meet the demand and the reorder point constraint.

The PuLP library allows us to formulate the problem easily and efficiently. Once the Linear Programming problem is defined, we call model.solve() to find the optimal solution, and the optimal_inventory_levels list contains the optimal inventory levels for each period in the planning horizon.

_Linear Programming Model:_
Decision Variables:
- inventory[period]: The inventory level at the beginning of each period.
- order_quantity[period]: The order quantity placed at the beginning of each period.

Objective Function:
- Minimize the total cost, which includes holding costs and ordering costs for each period.

Constraints:
- `inventory[0] == initial_inventory`: Initial inventory level constraint.
- `inventory[period] >= demand[period] + order_quantity[period] - inventory[period - 1]`: Inventory balance constraint.
- `inventory[period] >= reorder_point`: Reorder point constraint.
- `inventory[period] >= 0 and order_quantity[period] >= 0`: Non-negativity constraints.

Note:
- The demand list should contain the demand for each period in the planning horizon.
- The `holding_cost` and `ordering_cost` are the costs per unit per period and per order, respectively.
- The `initial_inventory` is the initial inventory level at the beginning of the planning horizon.
- The `reorder_point` is the inventory level at which a new order should be placed.
- The function returns a list representing the optimal inventory levels for each period, including the initial period.

> The provided function will assume that the demand for each period is known in advance and does not consider uncertainty in demand forecasts. Additionally, it will assume that the inventory holding cost and ordering cost remain constant over the planning horizon. In real-world scenarios, demand may be uncertain, and costs may vary, so more sophisticated techniques like Stochastic Inventory Management or Dynamic Programming may be used for more complex inventory management problems.


In [5]:
!pip install pulp

In [7]:
import pulp

@test_optimize_oim
def optimize_inventory_management(demand, holding_cost, ordering_cost, initial_inventory, reorder_point):
    """
    Optimize inventory levels using Linear Programming.
    """
    num_periods = len(demand)

    # Crear un problema de Programación Lineal (Minimización)
    model = pulp.LpProblem("Inventory_Optimization", pulp.LpMinimize)

    # Variables de decisión
    inventory = pulp.LpVariable.dicts("Inventory", range(num_periods + 1), lowBound=0, cat='Continuous')
    order_quantity = pulp.LpVariable.dicts("Order_Quantity", range(num_periods), lowBound=0, cat='Continuous')

    # Función objetivo: minimizar el costo total (almacenamiento + órdenes)
    model += pulp.lpSum([holding_cost * inventory[t] for t in range(1, num_periods + 1)]) + \
             pulp.lpSum([ordering_cost * order_quantity[t] for t in range(num_periods)])

    # Restricción de inventario inicial
    model += (inventory[0] == initial_inventory)

    # Restricciones para cada periodo
    for t in range(num_periods):
        # Restricción de balance de inventario: Inv_t = Inv_{t-1} + Orden_t - Demanda_t
        model += (inventory[t+1] == inventory[t] + order_quantity[t] - demand[t])

        # Restricción del punto de reorden (el inventario disponible debe cumplir con el punto de reorden mínimo)
        model += (inventory[t] >= reorder_point)

    # Resolver el problema de Programación Lineal
    model.solve(pulp.PULP_CBC_CMD(msg=False))

    # Extraer la solución óptima incluyendo el inventario inicial y de cada periodo
    optimal_inventory_levels = [inventory[t].varValue for t in range(num_periods + 1)]

    return optimal_inventory_levels

OIM Task  Failed

